## Building an Agent with Claude API

In [2]:
# Import libraries

from anthropic import Anthropic
from dotenv import load_dotenv

In [3]:
# Loading Anthropic API Key
load_dotenv()

True

## STEP 1: Creating a Client and asking a question

In [14]:
# Create an API Client
client = Anthropic()

# Defining parameters
model = "claude-haiku-4-5"
max_tokens=1000
prompt="Howdyyyy"

In [24]:
# Send a request
message = client.messages.create(
    model=model,
    max_tokens=max_tokens,
    messages=[{
        "role":"user",
        "content": prompt
    }

])

In [25]:
# Checking the entire output

message

Message(id='msg_01BDSuQuAYgyvvXYy36foSkj', container=None, content=[TextBlock(citations=None, text="Howdy! 👋 What's up? How can I help you today?", type='text')], model='claude-haiku-4-5-20251001', role='assistant', stop_details=None, stop_reason='end_turn', stop_sequence=None, type='message', usage=Usage(cache_creation=CacheCreation(ephemeral_1h_input_tokens=0, ephemeral_5m_input_tokens=0), cache_creation_input_tokens=0, cache_read_input_tokens=0, inference_geo='not_available', input_tokens=12, output_tokens=22, server_tool_use=None, service_tier='standard'))

In [6]:
# Retrieve the answer
message.content[0].text

"Howdy! 👋 How's it going? What can I help you with today?"

In [ ]:
# Try multiple messages
message = client.messages.create(
    model=model,
    max_tokens=max_tokens,
    messages=[{
        "role":"user",
        "content": "Hoowdyyy"
    },
    {
        "role":"user",
        "content": "Can you tell me what time is it?"
    }

])

## A new value under the messages lists provides additional context to the same prompt

In [8]:
# Retrieving the answer
message.content[0].text

'Howdy! 👋\n\nI don\'t have access to real-time information, so I can\'t tell you the current time. However, you can check the time by:\n\n- Looking at your device (phone, computer, watch)\n- Asking a voice assistant like Siri, Alexa, or Google Assistant\n- Searching "current time" online\n\nIs there anything else I can help you with?'

In [ ]:
# Trying user-assistant multi-conversation
# The LLM doesn't store the messages by default. 
# Here, I'm simulating how Claude would behave if we store interactions under the same message list.

message = client.messages.create(
    model=model,
    max_tokens=max_tokens,
    messages=[{
        "role":"user",
        "content": "What is an apple?"
    },
{
        "role":"assistant",
        "content": "An apple is a red fruit"
    },
{
        "role":"user",
        "content": "What about green ones?"
    }

])


In [10]:
# Retrieving the answer
message.content[0].text

"You're right to point that out! Apples come in various colors, including:\n\n- **Red** apples (like Red Delicious, Gala)\n- **Green** apples (like Granny Smith)\n- **Yellow** apples (like Golden Delicious)\n- **Mixed colors** (like Fuji or Honeycrisp)\n\nSo my initial answer was incomplete. Apples are fruits that grow on apple trees and come in multiple colors, each variety having different tastes and uses—some are sweeter, some more tart."

## STEP 2: Storing message interactions

In [ ]:
# Creating functions to maintain context for conversations

# messages it's a list that will be defined when calling the function
# add_user_message adds the user prompt do the message list whenever the user asks something
# add_assistant message does the same, but whenever the LLM answers something
# the chat() function only puts the request inside a function

def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)

def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)

def chat(messages):
    message = client.messages.create(
        model=model,
        max_tokens=max_tokens,
        messages=messages
    )
    return message.content[0].text

In [16]:
# Creating list - it will append the conversation history

messages = []

def conversation(messages):
    while True:
        try: 
            user_prompt = input("Prompt: ") ## input() is a function to prompt the user to insert a message
            if user_prompt.lower() == 'exit': # handling errors when prompting
                break
        except KeyboardInterrupt:
            print('Assistant: Goodbye')
            break
        except EOFError:
            print('Assistant: Goodbye')
            break
                                            ## executing the interaction flow
        add_user_message(messages, user_prompt)
        print(f"User: {user_prompt}")
        answer = chat(messages)
        print(f"Assistant: {answer}")
        add_assistant_message(messages, answer)



In [18]:
conversation(messages)

User: Could you list 5 types of apples to me?
Assistant: # 5 Types of Apples

1. **Gala** — Sweet and mild, with a thin skin. Great for eating fresh.

2. **Granny Smith** — Bright green, tart, and crisp. Popular for baking and cooking.

3. **Fuji** — Sweet, dense, and crispy. One of the most popular eating apples.

4. **Honeycrisp** — Very crisp and juicy with a balanced sweet-tart flavor. Excellent for fresh eating.

5. **Red Delicious** — Deep red, sweet with mild tartness. A classic eating apple.

Is there anything specific you'd like to know about these apples?
User: What about coconuts?
Assistant: # 5 Types of Coconuts

1. **Green Coconut** — Young coconuts with soft, jelly-like flesh inside. High in coconut water. Common in tropical regions for fresh drinking.

2. **Brown Coconut** — Mature coconuts with harder shell and denser white meat. Used for coconut milk, oil, and dried coconut products.

3. **Dwarf Coconut** — Smaller variety that matures quickly. Commonly grown in home g

## STEP 3: Tuning the parameters

In [19]:
# For agents, a good practice is to initialize the conversation with a system message 
#to give more context to the LLM about their role. Example: '''

messages = [
    {
        "role": "user", 
        "content": "You are a helpful assistant specialized in botanic."
    }
]
conversation(messages)

User: 
Assistant: # Hello! 🌿

I'm your botanical assistant, here to help you with:

- **Plant identification** - Describe a plant and I'll help identify it
- **Care & cultivation** - Watering, sunlight, soil, and growing tips
- **Plant biology** - Structure, growth, reproduction, and physiology
- **Ecology & uses** - Habitats, ecological roles, medicinal/culinary uses
- **Troubleshooting** - Pest problems, diseases, and plant health issues
- **Gardening advice** - From houseplants to outdoor gardens
- **Taxonomy & classification** - Plant families, species information

Feel free to ask me anything plant-related! Whether you're a gardening enthusiast, curious botanist, or simply looking to keep your houseplants healthy, I'm here to help. 🌱

**What would you like to know about plants today?**


In [20]:
''' Or the request can take a system parameter, where you input a system message.'''

# Send a request with a system prompt
system_prompt="You are a helpful assistant specialized in botanic."

message = client.messages.create(
    model=model,
    max_tokens=max_tokens,
    messages=[{
        "role":"user",
        "content": "Do you have any tip for me?"
    },],
    system=system_prompt
    )
message.content[0].text

"# Botanic Tips for You! 🌱\n\nHere are some helpful gardening and plant care tips:\n\n## General Plant Care\n- **Water wisely** – Most plants prefer soil that's moist but not waterlogged. Check soil moisture before watering\n- **Light matters** – Know your plant's light needs; too little causes weak growth, too much can scorch leaves\n- **Humidity helps** – Mist leaves occasionally or group plants together to increase humidity\n\n## Common Issues\n- **Yellowing leaves** – Usually overwatering; check drainage\n- **Pale leaves** – Often a sign of insufficient light\n- **Pest problems** – Inspect regularly and treat early with neem oil or insecticidal soap\n\n## Seasonal Tips\n- **Spring/Summer** – Increase watering and fertilize during growth season\n- **Fall/Winter** – Reduce watering and hold off on fertilizer as growth slows\n\n## General Success\n- Repot when roots emerge from drainage holes\n- Use well-draining soil appropriate for your plant type\n- Rotate plants quarterly for even

In [21]:
'''Testing different temperatures.
0: Deterministic output - tasks that are factual
1: Random output - tasks that are creative
'''

system_prompt="You are a helpful assistant specialized in botanic."

message = client.messages.create(
    model=model,
    max_tokens=max_tokens,
    messages=[{
        "role":"user",
        "content": "Do you have any tip for me?"
    },],
    system=system_prompt,
    temperature=1
    )
message.content[0].text

"# Tips for Botany & Plants 🌿\n\nI'd be happy to help! Here are some general tips depending on what you're interested in:\n\n## **For Plant Care:**\n- **Light**: Know your plant's needs—some thrive in shade, others need direct sun\n- **Watering**: Check soil moisture before watering; most plants prefer slightly dry to evenly moist\n- **Drainage**: Use pots with drainage holes to prevent root rot\n- **Humidity**: Mist tropical plants or group them together to increase humidity\n\n## **For Gardening:**\n- **Soil quality**: Invest in good potting mix or garden soil rich in organic matter\n- **Seasonal timing**: Plant in appropriate seasons for your climate zone\n- **Companion planting**: Some plants grow better together (like tomatoes with basil)\n\n## **For Plant Identification:**\n- Learn key features: leaf shape, color, texture, flowering patterns\n- Take clear photos from multiple angles\n- Note where you found it\n\n## **General Botany Tips:**\n- Observe plants in nature—it's the bes

In [22]:
'''Implementing Response Streaming
    It outputs the text while it is generate, instead all at once.
'''

stream = client.messages.stream(
    model=model,
    max_tokens=max_tokens,
    messages=[{
        "role":"user",
        "content": "Do you have any tip for me?"
    },],
    system=system_prompt,
    temperature=1,
    )

with stream as stream:
    for text in stream.text_stream:
        print(text, end="")

# Botanic Tips for You! 🌿

I'd be happy to help! Here are some general gardening and plant care tips:

## **Basic Plant Care**
- **Watering**: Most plants prefer soil that's moist but not waterlogged. Check soil before watering.
- **Light**: Know your plant's needs—some thrive in shade, others need full sun (6+ hours)
- **Drainage**: Use pots with drainage holes to prevent root rot

## **Common Beginner Tips**
- Start with hardy plants (pothos, snake plants, ZZ plants)
- Group plants with similar water needs together
- Repot when roots start circling the bottom
- Fertilize during growing season (spring/summer), not dormant seasons

## **Soil & Environment**
- Use quality potting mix appropriate for your plant type
- Maintain good air circulation to prevent fungal issues
- Most indoor plants appreciate 40-60% humidity

## **Troubleshooting**
- Yellow leaves = often overwatering
- Brown tips = underwatering or low humidity
- Pale leaves = insufficient light

---

**Do you have a specific

In [23]:
response = stream.get_final_message()
response

ParsedMessage(id='msg_01UviufLfbm2oiUtU78FYdTH', container=None, content=[ParsedTextBlock(citations=None, text="# Botanic Tips for You! 🌿\n\nI'd be happy to help! Here are some general gardening and plant care tips:\n\n## **Basic Plant Care**\n- **Watering**: Most plants prefer soil that's moist but not waterlogged. Check soil before watering.\n- **Light**: Know your plant's needs—some thrive in shade, others need full sun (6+ hours)\n- **Drainage**: Use pots with drainage holes to prevent root rot\n\n## **Common Beginner Tips**\n- Start with hardy plants (pothos, snake plants, ZZ plants)\n- Group plants with similar water needs together\n- Repot when roots start circling the bottom\n- Fertilize during growing season (spring/summer), not dormant seasons\n\n## **Soil & Environment**\n- Use quality potting mix appropriate for your plant type\n- Maintain good air circulation to prevent fungal issues\n- Most indoor plants appreciate 40-60% humidity\n\n## **Troubleshooting**\n- Yellow leave

## STEP 4: Creating Tools
The goal will be to create tools so Claude can send reminders

- Tool 1: Get the current date and time
- Tool 2: Calculate date time differences properly
- Tool 3: Set a reminder

In [27]:
from anthropic.types import ToolParam

def get_current_datetime(date_format="%Y-%m-%d %H:%M:%S"):
    from datetime import datetime 
    if not date_format:
        raise ValueError("date_format cannot be empty")
    
    return datetime.now().strftime(date_format)

## Defining the JSON schema for the function 
get_current_datetime__schema = ToolParam({
    "name": "get_current_datetime",
    "description": "Returns the current date and time formatted according to the specified format",
    "input_schema": {
        "type": "object",
        "properties": {
            "date_format": {
                "type": "string",
                "description": "A string specifying the format of the returned datetime. Uses Python's strftime format codes.",
                "default": "%Y-%m-%d %H:%M:%S"
            }
        },
        "required": []
    }
})


In [ ]:
stream = client.messages.stream(
    model=model,
    max_tokens=max_tokens,
    messages=[{
        "role":"user",
        "content": "What time is it now?"
    },],
    temperature=1,
    tools=[get_current_datetime__schema]
    )

with stream as stream:
    for text in stream.text_stream:
        print(text, end="")

In [31]:
response = stream.get_final_message()
response

ParsedMessage(id='msg_01SwbvA8V3qGjLuqW7cjEFP9', container=None, content=[ToolUseBlock(id='toolu_01A7mn9nFmiSyAgnDLXyJjrC', caller=DirectCaller(type='direct'), input={}, name='get_current_datetime', type='tool_use')], model='claude-haiku-4-5-20251001', role='assistant', stop_details=None, stop_reason='tool_use', stop_sequence=None, type='message', usage=Usage(cache_creation=CacheCreation(ephemeral_1h_input_tokens=0, ephemeral_5m_input_tokens=0), cache_creation_input_tokens=0, cache_read_input_tokens=0, inference_geo='not_available', input_tokens=621, output_tokens=39, server_tool_use=None, service_tier='standard'))

In [ ]:
tool = response.content[0].name
param = response.content[0].input

In [61]:
function = globals()[tool]
function()

'2026-05-29 17:35:34'